# Experimento Final — Knowledge Distillation

**MO434 - Aprendizado Profundo para Visao Computacional**

---

## Sobre este notebook

Este notebook implementa o **experimento final do projeto**, consolidando os resultados
das ablacoes documentadas em `guia_knowledge_distillation_t3.ipynb`.

Aquele notebook de estudo cobre:
- Secoes 1–3: datasets, teachers e Fase 1
- Secoes 4–6: arquitetura do student, funcoes de perda e Fase 2
- Secoes 7–9: Q5 (RKD), Fase 3 e eficiencia
- Secao 10: checklist de proximos passos

---

## Plano de experimentos

| # | Secao | O que responde |
|---|-------|---------------|
| 1 | Datasets | — |
| 2 | Fase 1 — todos os teachers, ambos os datasets | base para Q1 e entrega final |
| 3 | **Q1** — qual teacher transfere melhor? | 3 teachers × 2 datasets |
| 4 | **Q3 por teacher** — melhor encoder para cada teacher | entrega: best student por teacher |
| 5 | Treino definitivo — best student por teacher (50 ep) | Q1 + Q3 definitivos |
| 6 | **Q5** — MSE+CE vs RKD | literatura de KD |
| 7 | GFLOPs e eficiencia computacional | Q3 (saving GFLOPs e params) |
| 8 | Relatorio final consolidado | todas as questoes |

## Sintese das ablacoes (guia_knowledge_distillation_t3)

| Pergunta | Melhor | acc_test |
|----------|--------|---------|
| Q2 — target | post_gap | 0.2495 (2.5× > pre_gap) |
| Q4 — alpha | **0.5** | 0.2542 |

Essas conclusoes sao aplicadas diretamente: usamos `post_gap` e `alpha=0.5` em todos os
experimentos deste notebook.


---
## Secao 0 — Importacoes e Configuracao


In [8]:
# ── montar o Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# ── adicionar a pasta Project ao sys.path (para importar os .py locais) ───
import sys, os

PROJECT_PATH = '/content/drive/MyDrive/MO434-Deep-Learning/'
os.chdir(PROJECT_PATH)       # cwd = Project, então open('./data') funciona
sys.path.insert(0, PROJECT_PATH)  # Python encontra kd_utils.py, losses.py, etc.

print("Diretório de trabalho:", os.getcwd())

Diretório de trabalho: /content/drive/MyDrive/MO434-Deep-Learning


In [10]:
# instala fvcore para calculo de GFLOPs (tenta silenciosamente)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'fvcore', '-q'], capture_output=True)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.optim as optim

from kd_run import (
    set_seed, Timer,
    ImageTransforms, DatasetManager,
    TeacherWrapper,
    PlainCNNEncoder, DepthwiseCNNEncoder, MiniResNetEncoder,
    PreditorPostGAP, PreditorPreGAP, StudentModel,
    PerdaKD, PerdaRKD,
    Trainer, Evaluator,
    plotar_curvas, plotar_comparacao_mse_rkd,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'dispositivo: {device}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')
    print(f'memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


dispositivo: cuda
gpu: Tesla T4
memoria: 15.6 GB


In [11]:
# ── configuracoes globais ─────────────────────────────────────────────────────
SEED             = 42
BATCH_SIZE       = 32
NUM_WORKERS      = 2
STUDENT_CHANNELS = (32, 64, 128, 256)

# melhores hiperparametros das ablacoes (guia_knowledge_distillation_t3)
MELHOR_TARGET = 'post_gap'   # Q2: post_gap >> pre_gap
MELHOR_ALPHA  = 0.5          # Q4: melhor alpha entre 0, 0.5, 0.7, 1.0
EPOCHS_FASE1  = 15
EPOCHS_Q3     = 30           # ablacao de encoder (nao precisa ser definitiva)
EPOCHS_FINAL  = 50           # treino definitivo por teacher
LR            = 1e-3

TEACHERS  = ['resnet50', 'vgg16', 'convnext_small']
DATASETS_NOMES = ['flowers102', 'pets']
ENCODERS  = [
    ('PlainCNN',    PlainCNNEncoder),
    ('Depthwise',   DepthwiseCNNEncoder),
    ('MiniResNet',  MiniResNetEncoder),
]

trainer   = Trainer(device=device, seed=SEED)
evaluator = Evaluator(device=device)
os.makedirs('checkpoints', exist_ok=True)

# ── helper: GFLOPs via fvcore (fallback: None) ────────────────────────────────
def gflops_params(model, input_size=(1, 3, 224, 224)):
    model.eval()
    dummy = torch.randn(*input_size)
    params = sum(p.numel() for p in model.parameters()) / 1e6
    try:
        from fvcore.nn import FlopCountAnalysis
        gf = FlopCountAnalysis(model.cpu(), dummy).total() / 1e9
        return round(gf, 3), round(params, 3)
    except Exception:
        return None, round(params, 3)

print('configuracoes prontas.')


semente fixada: 42 (random, numpy, torch, cuda)
configuracoes prontas.


---
## Secao 1 — Datasets

- **Flowers-102**: 102 classes de flores, ~8K imagens, splits oficiais
- **Oxford-IIIT-Pet**: 37 racas de caes e gatos, ~7K imagens


In [12]:
with Timer('carregamento dos datasets'):
    dm = DatasetManager(data_root='./data', batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    DATASETS = dm.load_all()

for nome in DATASETS_NOMES:
    print(f'{nome:12s}: {DATASETS[nome]["n_classes"]} classes')


100%|██████████| 345M/345M [00:16<00:00, 20.9MB/s]
100%|██████████| 502/502 [00:00<00:00, 282kB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 40.9MB/s]


flowers-102 -> treino: 1,020 | val: 1,020 | teste: 6,149


100%|██████████| 792M/792M [00:40<00:00, 19.8MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 12.7MB/s]


oxford-pets -> treino: 2,944 | val: 736 | teste: 3,669
[timer] carregamento dos datasets                     -> 9 min 33.5 s
flowers102  : 102 classes
pets        : 37 classes


---
## Secao 2 — Fase 1: Treino do Classificador (todos os teachers)

Para cada teacher e cada dataset, congela o encoder e treina apenas o classificador.
O classificador salvo aqui sera reutilizado **exatamente** na Fase 3.

Total: 3 teachers × 2 datasets = **6 runs**.


In [13]:
# ── Fase 1: 3 teachers × 2 datasets ─────────────────────────────────────────
# resultados_fase1[teacher][dataset] = {'teacher_obj', 'acc_test'}
resultados_fase1 = {t: {} for t in TEACHERS}

for nome_teacher in TEACHERS:
    for dataset_nome in DATASETS_NOMES:
        n_classes = DATASETS[dataset_nome]['n_classes']
        print(f'\n{"-"*55}')
        print(f' {nome_teacher} | {dataset_nome} ({n_classes} classes)')
        print(f'{"-"*55}')

        teacher = TeacherWrapper(nome_teacher, n_classes=n_classes).to(device)
        teacher.freeze_encoder()

        params_tr = [p for p in teacher.parameters() if p.requires_grad]
        optimizer = optim.Adam(params_tr, lr=LR, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FASE1)

        with Timer(f'fase1 {nome_teacher}/{dataset_nome}'):
            hist = trainer.treinar_fase1(
                model        = teacher,
                loader_train = DATASETS[dataset_nome]['train'],
                loader_val   = DATASETS[dataset_nome]['val'],
                optimizer    = optimizer,
                scheduler    = scheduler,
                n_epochs     = EPOCHS_FASE1,
                descricao    = f'{nome_teacher}_{dataset_nome}',
            )

        acc_test = trainer.avaliar_loader(teacher, DATASETS[dataset_nome]['test'])
        print(f'  acc_test: {acc_test:.4f}')

        torch.save(teacher.classifier.state_dict(),
                   f'checkpoints/classifier_{nome_teacher}_{dataset_nome}.pth')

        resultados_fase1[nome_teacher][dataset_nome] = {
            'teacher_obj': teacher,
            'acc_test':    acc_test,
        }

        # libera GPU apos salvar (sera restaurado quando necessario)
        teacher.cpu()

print('\nFase 1 concluida.')



-------------------------------------------------------
 resnet50 | flowers102 (102 classes)
-------------------------------------------------------
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 136MB/s]


encoder resnet50 congelado.


  epoca   1 | loss_tr=4.3601 acc_tr=0.163 | loss_vl=3.7976 acc_vl=0.590 | gap=-0.427


  epoca   5 | loss_tr=1.5228 acc_tr=0.971 | loss_vl=1.9001 acc_vl=0.835 | gap=+0.136


...

  epoca  15 | loss_tr=0.3617 acc_tr=0.975 | loss_vl=0.6119 acc_vl=0.900 | gap=+0.075
  melhor acc validacao: 0.9037 (epoca 11)
[timer] fase1 convnext_small/flowers102               -> 6 min 11.7 s
  acc_test: 0.8679

-------------------------------------------------------
 convnext_small | pets (37 classes)
-------------------------------------------------------
encoder convnext_small congelado.


  epoca   1 | loss_tr=1.1871 acc_tr=0.765 | loss_vl=0.2866 acc_vl=0.932 | gap=-0.167


  epoca   5 | loss_tr=0.1304 acc_tr=0.969 | loss_vl=0.1705 acc_vl=0.951 | gap=+0.018


  epoca  10 | loss_tr=0.0784 acc_tr=0.980 | loss_vl=0.1206 acc_vl=0.957 | gap=+0.023


  epoca  15 | loss_tr=0.0688 acc_tr=0.986 | loss_vl=0.1206 acc_vl=0.955 | gap=+0.031
  melhor acc validacao: 0.9701 (epoca 12)
[timer] fase1 convnext_small/pets                     -> 10 min 59.4 s
  acc_test: 0.9329

Fase 1 concluida.


---
## Secao 3 — Q1: Qual teacher transfere melhor?

Comparamos os 3 teachers usando o melhor student das ablacoes
(PlainCNN, post_gap, alpha=0.5) em **ambos os datasets**.


In [ ]:
# ── Q1: 3 teachers × 2 datasets (PlainCNN, post_gap, alpha=0.5, 30 ep) ────────
resultados_q1 = []

for nome_teacher in TEACHERS:
    for dataset_nome in DATASETS_NOMES:
        teacher = resultados_fase1[nome_teacher][dataset_nome]['teacher_obj'].to(device)

        enc     = PlainCNNEncoder(STUDENT_CHANNELS)
        pred    = PreditorPostGAP(enc.out_dim, teacher.feat_dim)
        student = StudentModel(enc, pred).to(device)

        perda_fn = PerdaKD(alpha=MELHOR_ALPHA)
        with Timer(f'Q1 {nome_teacher}/{dataset_nome}'):
            hist, _ = trainer.treinar_fase2(
                student      = student,
                teacher      = teacher,
                loader_train = DATASETS[dataset_nome]['train'],
                loader_val   = DATASETS[dataset_nome]['val'],
                perda_fn     = perda_fn,
                n_epochs     = EPOCHS_Q3,
                lr           = LR,
                modo_target  = MELHOR_TARGET,
                descricao    = f'q1_{nome_teacher}_{dataset_nome}',
            )

        acc_test, acc_top5 = evaluator.avaliar_fase3(
            student, teacher, DATASETS[dataset_nome]['test'], MELHOR_TARGET)

        resultados_q1.append({
            'teacher':  nome_teacher,
            'dataset':  dataset_nome,
            'acc_test': acc_test,
            'acc_top5': acc_top5,   # relevante especialmente para flowers102 (102 classes)
        })
        teacher.cpu(); student.cpu()

df_q1 = pd.DataFrame(resultados_q1)

print('\n' + '='*60)
print(' Q1 — Qual teacher transfere melhor?')
print('='*60)
print('\n  Top-1 accuracy:')
piv1 = df_q1.pivot(index='teacher', columns='dataset', values='acc_test')
print(piv1.to_string())

# top-5 (flowers102 tem 102 classes: top-5 e mais informativo)
df_top5 = df_q1[df_q1['acc_top5'].notna()]
if not df_top5.empty:
    print('\n  Top-5 accuracy:')
    piv5 = df_top5.pivot(index='teacher', columns='dataset', values='acc_top5')
    print(piv5.to_string())

# grafico Q1
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ds in zip(axes, DATASETS_NOMES):
    dados = df_q1[df_q1['dataset'] == ds].sort_values('acc_test', ascending=True)
    ax.barh(dados['teacher'], dados['acc_test'], color='steelblue', alpha=0.85,
            label='Top-1')
    # top-5 como marcador adicional
    top5_vals = dados['acc_top5'].values
    if any(v is not None for v in top5_vals):
        top5_clean = [v if v else 0 for v in top5_vals]
        ax.barh(dados['teacher'], top5_clean, color='lightsteelblue', alpha=0.4,
                label='Top-5')
    ax.set_title(f'Q1 — {ds}', fontsize=11)
    ax.set_xlabel('acuracia'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='x')
    for i, (_, row) in enumerate(dados.iterrows()):
        ax.text(row['acc_test'] + 0.002, i, f"{row['acc_test']:.3f}", va='center', fontsize=9)
plt.suptitle('Q1 — Transferencia por Teacher (Top-1 escuro, Top-5 claro)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


  ep.  1 | mse=0.0369 | acc_tr=0.011 acc_vl=0.017 | gap=-0.006


  ep.  5 | mse=0.0333 | acc_tr=0.082 acc_vl=0.090 | gap=-0.008


  ep.  8 | mse=0.0328 | acc_tr=0.134 acc_vl=0.125 | gap=+0.009  [OVERFITTING: val_loss subindo 3x seguidas]


  ep. 10 | mse=0.0326 | acc_tr=0.164 acc_vl=0.150 | gap=+0.014


  ep. 15 | mse=0.0318 | acc_tr=0.247 acc_vl=0.198 | gap=+0.049


...

  ep. 30 | mse=0.0513 | acc_tr=0.360 acc_vl=0.306 | gap=+0.054
  melhor acc validacao: 0.3125 (epoca 28)
[timer] Q1 resnet50/pets                              -> 19 min 20.2 s


  ep.  1 | mse=0.1505 | acc_tr=0.026 acc_vl=0.033 | gap=-0.008


  ep.  5 | mse=0.1158 | acc_tr=0.104 acc_vl=0.099 | gap=+0.005


  ep. 10 | mse=0.1079 | acc_tr=0.170 acc_vl=0.133 | gap=+0.037


  ep. 15 | mse=0.1024 | acc_tr=0.227 acc_vl=0.168 | gap=+0.059


---
## Secao 4 — Q3 por Teacher: Melhor Encoder para cada Teacher

Para cada teacher identificamos qual dos 3 encoders produz a maior acuracia no
Flowers-102 (flores: dominio mais discriminativo, ideal para comparar arquiteturas).

**Entrega exigida pelo enunciado**: "provide the notebooks/Python scripts with the
best student architecture for each teacher."

Total: 3 teachers × 3 encoders = **9 runs** (30 epocas cada).


In [ ]:
# ── Q3: melhor encoder por teacher (flowers102, post_gap, alpha=0.5, 30 ep) ───
resultados_q3 = []

for nome_teacher in TEACHERS:
    teacher = resultados_fase1[nome_teacher]['flowers102']['teacher_obj'].to(device)

    for enc_nome, EncoderCls in ENCODERS:
        enc     = EncoderCls(STUDENT_CHANNELS)
        pred    = PreditorPostGAP(enc.out_dim, teacher.feat_dim)
        student = StudentModel(enc, pred).to(device)
        params_s = sum(p.numel() for p in student.parameters()) / 1e6

        perda_fn = PerdaKD(alpha=MELHOR_ALPHA)
        with Timer(f'Q3 {nome_teacher}/{enc_nome}'):
            hist, melhor_acc_val = trainer.treinar_fase2(
                student      = student,
                teacher      = teacher,
                loader_train = DATASETS['flowers102']['train'],
                loader_val   = DATASETS['flowers102']['val'],
                perda_fn     = perda_fn,
                n_epochs     = EPOCHS_Q3,
                lr           = LR,
                modo_target  = MELHOR_TARGET,
                descricao    = f'q3_{nome_teacher}_{enc_nome}',
            )

        acc_test, _ = evaluator.avaliar_fase3(
            student, teacher, DATASETS['flowers102']['test'], MELHOR_TARGET)

        resultados_q3.append({
            'teacher':   nome_teacher,
            'encoder':   enc_nome,
            'acc_test':  acc_test,
            'params_M':  params_s,
        })
        student.cpu()

    teacher.cpu()

df_q3 = pd.DataFrame(resultados_q3)
print('\n' + '='*60)
print(' Q3 — Melhor encoder por teacher (flowers102)')
print('='*60)
for t in TEACHERS:
    sub = df_q3[df_q3['teacher'] == t].sort_values('acc_test', ascending=False)
    print(f'\n  {t}')
    print(sub[['encoder','acc_test','params_M']].to_string(index=False))

# identifica best encoder por teacher
best_encoder_por_teacher = (
    df_q3.sort_values('acc_test', ascending=False)
         .groupby('teacher')
         .first()['encoder']
         .to_dict()
)
print('\nMelhor encoder por teacher:')
for t, enc in best_encoder_por_teacher.items():
    print(f'  {t:20s} -> {enc}')

# mapa nome -> classe
ENCODER_CLASSES = {
    'PlainCNN':   PlainCNNEncoder,
    'Depthwise':  DepthwiseCNNEncoder,
    'MiniResNet': MiniResNetEncoder,
}

# grafico Q3
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, t in zip(axes, TEACHERS):
    sub = df_q3[df_q3['teacher'] == t].sort_values('acc_test', ascending=True)
    cores = ['#e74c3c' if enc == best_encoder_por_teacher[t] else 'steelblue'
             for enc in sub['encoder']]
    ax.barh(sub['encoder'], sub['acc_test'], color=cores, alpha=0.85)
    ax.set_title(f'{t}', fontsize=10)
    ax.set_xlabel('acc_test'); ax.grid(True, alpha=0.3, axis='x')
plt.suptitle('Q3 — Melhor encoder por teacher (vermelho = melhor)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


---
## Secao 5 — Treino Definitivo: Best Student por Teacher (50 epocas)

Usa o melhor encoder identificado na Secao 4 para cada teacher,
treinando com 50 epocas em **ambos os datasets**.

Os checkpoints salvos aqui sao a **entrega final** do projeto.


In [ ]:
# ── Treino definitivo: best encoder por teacher, 50 ep, 2 datasets ───────────
resultados_final = {}

for nome_teacher in TEACHERS:
    resultados_final[nome_teacher] = {}
    EncoderCls = ENCODER_CLASSES[best_encoder_por_teacher[nome_teacher]]

    for dataset_nome in DATASETS_NOMES:
        teacher = resultados_fase1[nome_teacher][dataset_nome]['teacher_obj'].to(device)

        enc     = EncoderCls(STUDENT_CHANNELS)
        pred    = PreditorPostGAP(enc.out_dim, teacher.feat_dim)
        student = StudentModel(enc, pred).to(device)
        params_s = sum(p.numel() for p in student.parameters()) / 1e6

        print(f'\n{"-"*60}')
        print(f' {nome_teacher} | {best_encoder_por_teacher[nome_teacher]} | {dataset_nome}')
        print(f' params: {params_s:.2f}M')
        print(f'{"-"*60}')

        perda_fn = PerdaKD(alpha=MELHOR_ALPHA)
        with Timer(f'final {nome_teacher}/{dataset_nome}'):
            hist, melhor_acc_val = trainer.treinar_fase2(
                student      = student,
                teacher      = teacher,
                loader_train = DATASETS[dataset_nome]['train'],
                loader_val   = DATASETS[dataset_nome]['val'],
                perda_fn     = perda_fn,
                n_epochs     = EPOCHS_FINAL,
                lr           = LR,
                modo_target  = MELHOR_TARGET,
                descricao    = f'final_{nome_teacher}_{dataset_nome}',
            )

        acc_test, acc_top5 = evaluator.avaliar_fase3(
            student, teacher, DATASETS[dataset_nome]['test'], MELHOR_TARGET)

        print(f'  melhor acc val : {melhor_acc_val:.4f}')
        print(f'  acc_test (F3)  : {acc_test:.4f}')
        if acc_top5:
            print(f'  acc_top5       : {acc_top5:.4f}')

        ckpt = f'checkpoints/best_student_{nome_teacher}_{dataset_nome}.pth'
        torch.save(student.state_dict(), ckpt)
        print(f'  checkpoint     : {ckpt}')

        resultados_final[nome_teacher][dataset_nome] = {
            'student_obj':    student,
            'historico':      hist,
            'melhor_acc_val': melhor_acc_val,
            'acc_test':       acc_test,
            'acc_top5':       acc_top5,
            'params_M':       params_s,
            'encoder':        best_encoder_por_teacher[nome_teacher],
        }

        plotar_curvas(hist, titulo=f'{nome_teacher} / {dataset_nome} (50 ep)')
        student.cpu(); teacher.cpu()


---
## Secao 6 — Q5: MSE+CE vs RKD

Compara os dois esquemas de perda com o melhor student do resnet50:

| Metodo | Formula | O que alinha |
|--------|---------|-------------|
| **MSE+CE** (baseline) | α·L_MSE + (1-α)·L_CE | valores absolutos das features |
| **RKD+CE** | L_RKD + 0.3·L_CE | distancias par-a-par (invariante a escala) |

**Hipotese**: RKD pode superar MSE em *capacity mismatch* (student muito menor que teacher).

Referencia: Park et al., CVPR 2019.


In [ ]:
# ── Q5: MSE baseline vs RKD (resnet50, flowers102, best encoder, 30 ep) ───────
teacher_q5 = resultados_fase1['resnet50']['flowers102']['teacher_obj'].to(device)

with Timer('Q5 — MSE vs RKD'):
    resultados_q5 = trainer.comparar_mse_vs_rkd(
        teacher          = teacher_q5,
        datasets         = DATASETS,
        teacher_nome     = 'resnet50',
        dataset_nome     = 'flowers102',
        n_epochs         = 30,
        lr               = LR,
        student_channels = STUDENT_CHANNELS,
    )

print('\n' + '='*55)
print(' Q5 — MSE+CE baseline vs RKD')
print('='*55)
for nome, res in resultados_q5.items():
    print(f'  {nome:20s}  melhor_acc_val = {res["melhor_acc"]:.4f}')

plotar_comparacao_mse_rkd(resultados_q5, 'resnet50', 'flowers102')
teacher_q5.cpu()


---
## Secao 7 — GFLOPs e Eficiencia Computacional

O enunciado exige que o student **salve GFLOPs e parametros consideravelmente**
comparado ao teacher.


In [ ]:
# ── tabela de eficiencia: GFLOPs e parametros ─────────────────────────────────
linhas_eff = []

# teachers
for nome_teacher in TEACHERS:
    t = resultados_fase1[nome_teacher]['flowers102']['teacher_obj']
    gf, pr = gflops_params(t)
    linhas_eff.append({
        'modelo': f'{nome_teacher} (teacher)', 'dataset': '—',
        'gflops': gf, 'params_M': pr, 'tipo': 'teacher',
    })

# best students (flowers102, que e onde Q3 foi medido)
for nome_teacher in TEACHERS:
    res = resultados_final[nome_teacher]['flowers102']
    s   = res['student_obj']
    gf, pr = gflops_params(s)
    linhas_eff.append({
        'modelo': f'{res["encoder"]} (student/{nome_teacher})', 'dataset': 'flowers102',
        'gflops': gf, 'params_M': pr, 'tipo': 'student',
    })

df_eff = pd.DataFrame(linhas_eff)

print('Eficiencia Computacional')
print()
print(f'{"modelo":42s} {"gflops":>8s} {"params_M":>10s}')
print('─' * 65)

teacher_params = {}
for _, row in df_eff.iterrows():
    gf_str = f"{row['gflops']:.3f}" if row['gflops'] else 'n/d'
    pr_str = f"{row['params_M']:.2f}M"
    print(f"  {row['modelo']:40s} {gf_str:>8s} {pr_str:>10s}")
    if row['tipo'] == 'teacher':
        t_nome = row['modelo'].split(' ')[0]
        teacher_params[t_nome] = row['params_M']

# reducao relativa
print()
print('Reducao de parametros (student / teacher):')
for nome_teacher in TEACHERS:
    res = resultados_final[nome_teacher]['flowers102']
    reducao = res['params_M'] / teacher_params.get(nome_teacher, 1) * 100
    print(f'  {nome_teacher:20s}  {res["encoder"]:12s}  {res["params_M"]:.2f}M  ({reducao:.1f}%)')


---
## Secao 8 — Relatorio Final Consolidado

Tabela unica com os resultados de todas as questoes.


In [ ]:
# ── tabela consolidada final ──────────────────────────────────────────────────
linhas_rel = []

# Q1 + treino definitivo: best student por teacher × dataset
for nome_teacher in TEACHERS:
    for dataset_nome in DATASETS_NOMES:
        res = resultados_final[nome_teacher][dataset_nome]
        t_params = sum(
            p.numel() for p in resultados_fase1[nome_teacher][dataset_nome]['teacher_obj'].parameters()
        ) / 1e6
        linhas_rel.append({
            'questao':   'Q1/Q3 — definitivo',
            'teacher':   nome_teacher,
            'encoder':   res['encoder'],
            'dataset':   dataset_nome,
            'alpha':     MELHOR_ALPHA,
            'perda':     'MSE+CE',
            'acc_top1':  res['acc_test'],
            'acc_top5':  res['acc_top5'] if res['acc_top5'] else '—',
            'params_M':  res['params_M'],
            'reducao_%': round(res['params_M'] / t_params * 100, 1),
        })

# Q5
for nome_perda, res in resultados_q5.items():
    t_params = sum(
        p.numel() for p in resultados_fase1['resnet50']['flowers102']['teacher_obj'].parameters()
    ) / 1e6
    s_params = resultados_final['resnet50']['flowers102']['params_M']
    linhas_rel.append({
        'questao':   'Q5',
        'teacher':   'resnet50',
        'encoder':   best_encoder_por_teacher['resnet50'],
        'dataset':   'flowers102',
        'alpha':     '—',
        'perda':     nome_perda,
        'acc_top1':  res['melhor_acc'],
        'acc_top5':  '—',           # comparar_mse_vs_rkd nao retorna top5
        'params_M':  s_params,
        'reducao_%': round(s_params / t_params * 100, 1),
    })

df_rel = pd.DataFrame(linhas_rel)

print('=' * 85)
print(' RELATORIO FINAL — Knowledge Distillation (MO434)')
print('=' * 85)
print(df_rel[['questao','teacher','encoder','dataset','perda','alpha',
              'acc_top1','acc_top5','params_M','reducao_%']].to_string(index=False))

# ── Q1 pivot top-1 e top-5 ────────────────────────────────────────────────────
print('\n--- Q1: acc_top1 por teacher × dataset ---')
piv1 = df_rel[df_rel['questao'] == 'Q1/Q3 — definitivo'].pivot_table(
    index='teacher', columns='dataset', values='acc_top1')
print(piv1.to_string())

# top-5 (flowers102: 102 classes — top-5 e muito informativo)
df_t5 = df_rel[
    (df_rel['questao'] == 'Q1/Q3 — definitivo') &
    (df_rel['acc_top5'] != '—')
].copy()
df_t5['acc_top5'] = pd.to_numeric(df_t5['acc_top5'], errors='coerce')
if not df_t5.empty and df_t5['acc_top5'].notna().any():
    print('\n--- Q1: acc_top5 por teacher × dataset ---')
    piv5 = df_t5.pivot_table(index='teacher', columns='dataset', values='acc_top5')
    print(piv5.to_string())

# ── Q3 resumo ─────────────────────────────────────────────────────────────────
print('\n--- Q3: melhor encoder por teacher (flowers102, 50 ep) ---')
print(f'  {"teacher":20s} {"encoder":12s} {"acc_top1":>9s} {"acc_top5":>9s} {"params_M":>9s} {"reducao":>8s}')
print('  ' + '-'*72)
for t in TEACHERS:
    enc = best_encoder_por_teacher[t]
    res = resultados_final[t]['flowers102']
    top5_str = f"{res['acc_top5']:.4f}" if res['acc_top5'] else '  —   '
    reducao  = round(res['params_M'] / sum(
        p.numel() for p in resultados_fase1[t]['flowers102']['teacher_obj'].parameters()
    ) * 1e-6 * 100, 1)
    print(f"  {t:20s} {enc:12s} {res['acc_test']:>9.4f} {top5_str:>9s} {res['params_M']:>7.2f}M {reducao:>7.1f}%")

# ── Q5 ────────────────────────────────────────────────────────────────────────
print('\n--- Q5: MSE+CE vs RKD ---')
q5_tab = df_rel[df_rel['questao'] == 'Q5'][['perda', 'acc_top1']]
print(q5_tab.to_string(index=False))


In [ ]:
# ── graficos finais ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# --- Q1: acc por teacher × dataset ---
ax = axes[0, 0]
piv_plot = df_rel[df_rel['questao'] == 'Q1/Q3 — definitivo'].pivot_table(
    index='teacher', columns='dataset', values='acc_test')
x = range(len(piv_plot))
w = 0.35
ax.bar([xi - w/2 for xi in x], piv_plot.get('flowers102', [0]*3),
       width=w, label='flowers102', color='steelblue', alpha=0.85)
ax.bar([xi + w/2 for xi in x], piv_plot.get('pets', [0]*3),
       width=w, label='pets', color='darkorange', alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(piv_plot.index, rotation=10, fontsize=9)
ax.set_title('Q1 — Teacher × Dataset', fontsize=11)
ax.set_ylabel('acc_test'); ax.legend(); ax.grid(True, alpha=0.3, axis='y')

# --- Q3: encoder por teacher (flowers102) ---
ax = axes[0, 1]
sub_q3 = df_q3.copy()
cores_enc = {'PlainCNN': '#3498db', 'Depthwise': '#2ecc71', 'MiniResNet': '#e74c3c'}
for i, t in enumerate(TEACHERS):
    d = sub_q3[sub_q3['teacher'] == t].sort_values('encoder')
    ax.scatter([i]*len(d), d['acc_test'],
               c=[cores_enc.get(enc, 'gray') for enc in d['encoder']],
               s=120, zorder=5)
ax.set_xticks(range(len(TEACHERS))); ax.set_xticklabels(TEACHERS, rotation=10, fontsize=9)
ax.set_title('Q3 — Encoder por Teacher (flowers102)', fontsize=11)
ax.set_ylabel('acc_test'); ax.grid(True, alpha=0.3)
# legenda manual
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=e) for e, c in cores_enc.items()], fontsize=8)

# --- Q3: acuracia vs params ---
ax = axes[0, 2]
sub_def = df_rel[df_rel['questao'] == 'Q1/Q3 — definitivo']
for _, row in sub_def[sub_def['dataset'] == 'flowers102'].iterrows():
    ax.scatter(row['params_M'], row['acc_test'], s=130, zorder=5)
    ax.annotate(f"{row['teacher'][:3]}\n{row['encoder'][:5]}",
                (row['params_M'], row['acc_test']),
                textcoords='offset points', xytext=(5,3), fontsize=7)
ax.set_title('Q3 — Acuracia vs Params (flowers102)', fontsize=11)
ax.set_xlabel('params student (M)'); ax.set_ylabel('acc_test'); ax.grid(True, alpha=0.3)

# --- Q5: MSE vs RKD ---
ax = axes[1, 0]
q5_plot = df_rel[df_rel['questao'] == 'Q5']
cores_q5 = {'MSE_baseline': '#2ecc71', 'RKD': '#e74c3c'}
bars = ax.bar(q5_plot['perda'], q5_plot['acc_test'],
              color=[cores_q5.get(p, 'steelblue') for p in q5_plot['perda']],
              alpha=0.85, width=0.4)
for bar, (_, row) in zip(bars, q5_plot.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{row['acc_test']:.3f}", ha='center', fontsize=10, fontweight='bold')
ax.set_title('Q5 — MSE+CE vs RKD\n(resnet50, flowers102)', fontsize=11)
ax.set_ylabel('melhor acc_val'); ax.grid(True, alpha=0.3, axis='y')

# --- curvas de convergencia: best students (flowers102) ---
ax = axes[1, 1]
cores_t = {'resnet50': 'steelblue', 'vgg16': 'darkorange', 'convnext_small': 'green'}
for t in TEACHERS:
    hist = resultados_final[t]['flowers102']['historico']
    ep = range(len(hist['acc_vl']))
    ax.plot(ep, hist['acc_vl'], label=t, color=cores_t[t], linewidth=2)
ax.set_title('Convergencia val — flowers102 (50 ep)', fontsize=11)
ax.set_xlabel('epoca'); ax.set_ylabel('acc_val'); ax.legend(); ax.grid(True, alpha=0.3)

# --- validacao cruzada: flowers102 vs pets por teacher ---
ax = axes[1, 2]
sub_cross = df_rel[df_rel['questao'] == 'Q1/Q3 — definitivo']
piv_cross = sub_cross.pivot_table(index='teacher', columns='dataset', values='acc_test')
x2 = range(len(piv_cross))
ax.bar([xi - w/2 for xi in x2], piv_cross.get('flowers102', [0]*3),
       width=w, label='flowers102', color='steelblue', alpha=0.85)
ax.bar([xi + w/2 for xi in x2], piv_cross.get('pets', [0]*3),
       width=w, label='pets', color='darkorange', alpha=0.85)
ax.set_xticks(list(x2)); ax.set_xticklabels(piv_cross.index, rotation=10, fontsize=9)
ax.set_title('Validacao cruzada (50 ep)', fontsize=11)
ax.set_ylabel('acc_test'); ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Experimento Final — Knowledge Distillation (MO434)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Timer.resumo()


---
## Secao 9 — Analise Qualitativa

O enunciado exige *"quantitative and qualitative results on each dataset, with illustrations"*.

### 9.1 Exemplos de predicao: Teacher vs Student

Cada imagem mostra a predicao do teacher (T) e do student (S) com a classe verdadeira (gt).
A borda indica o resultado combinado:

| Borda | Significado |
|-------|-------------|
| **Verde** | Ambos corretos |
| **Laranja** | Apenas teacher correto (student errou) |
| **Vermelho** | Ambos errados |

### 9.2 t-SNE do espaco de features

Compara a geometria do espaco de representacoes do teacher (features post-GAP reais)
com a do student (predicoes do preditor). Clusters similares indicam boa transferencia:
o student aprendeu a organizar o espaco de features de forma parecida com o teacher.


In [ ]:
# ── 9.1 Exemplos de predicao: Teacher vs Student ─────────────────────────────
@torch.no_grad()
def visualizar_exemplos(student, teacher, loader_test, n_imgs=16,
                        dataset_nome='flowers102'):
    """
    Mostra n_imgs imagens do conjunto de teste com as predicoes do teacher e do student.

    Borda verde  = ambos corretos
    Borda laranja = so teacher correto (student errou — gap de transferencia)
    Borda vermelha = ambos errados
    """
    student.eval(); teacher.eval()

    # coleta um batch do teste
    imgs, labels = next(iter(loader_test))
    imgs, labels = imgs[:n_imgs].to(device), labels[:n_imgs].to(device)

    # predicoes do teacher (features reais + classificador)
    feat_t   = teacher.get_post_gap(imgs)
    logits_t = teacher.classifier(feat_t)
    preds_t  = logits_t.argmax(1)

    # predicoes do student (preditor -> mesmo classificador do teacher)
    pred_s   = student(imgs)
    logits_s = teacher.classifier(pred_s)
    preds_s  = logits_s.argmax(1)

    # desfaz normalizacao imagenet para exibir
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(3, 1, 1)

    n_cols = 8
    n_rows = (n_imgs + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 2.2, n_rows * 2.8))
    axes = axes.flatten()

    for i in range(n_imgs):
        ax  = axes[i]
        img = (imgs[i] * std + mean).clamp(0, 1).permute(1, 2, 0).cpu().numpy()
        ax.imshow(img)

        true  = labels[i].item()
        t_ok  = preds_t[i].item() == true
        s_ok  = preds_s[i].item() == true

        # borda indica resultado
        cor = 'green' if (t_ok and s_ok) else ('darkorange' if t_ok else 'red')
        for sp in ax.spines.values():
            sp.set_edgecolor(cor)
            sp.set_linewidth(3)

        # titulo: predicao teacher | predicao student | verdadeiro
        icone_t = '✓' if t_ok else '✗'
        icone_s = '✓' if s_ok else '✗'
        ax.set_title(f'T{icone_t}:{preds_t[i].item()}  S{icone_s}:{preds_s[i].item()}'
                     f'\ngt:{true}', fontsize=7)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for i in range(n_imgs, len(axes)):
        axes[i].axis('off')

    plt.suptitle(
        f'Predicoes Teacher vs Student | {dataset_nome}\n'
        f'T=teacher  S=student  gt=classe real  '
        f'verde=ambos certos  laranja=só teacher  vermelho=ambos errados',
        fontsize=9, y=1.01)
    plt.tight_layout()
    plt.show()


# ── executa para o melhor teacher (resnet50) nos dois datasets ────────────────
for ds in DATASETS_NOMES:
    t_viz = resultados_fase1['resnet50'][ds]['teacher_obj'].to(device)
    s_viz = resultados_final['resnet50'][ds]['student_obj'].to(device)
    visualizar_exemplos(s_viz, t_viz, DATASETS[ds]['test'],
                        n_imgs=16, dataset_nome=ds)
    t_viz.cpu(); s_viz.cpu()


In [ ]:
# ── 9.2 t-SNE: espaco de features do Teacher vs Student ──────────────────────
@torch.no_grad()
def plotar_tsne(student, teacher, loader_test, n_amostras=500,
                dataset_nome='flowers102'):
    """
    Projeta em 2D (via PCA + t-SNE) as features do teacher e as predicoes
    do student, colorindo cada ponto pela classe verdadeira.

    Clusters similares entre os dois mapas indicam que o student aprendeu
    a geometria relacional do espaco do teacher (objetivo do KD).
    """
    try:
        from sklearn.manifold import TSNE
        from sklearn.decomposition import PCA
    except ImportError:
        print('sklearn nao instalado. Execute: pip install scikit-learn')
        return

    student.eval(); teacher.eval()

    feats_t, feats_s, lbls = [], [], []
    total = 0

    for imgs, labels in loader_test:
        if total >= n_amostras:
            break
        imgs = imgs.to(device)
        feats_t.append(teacher.get_post_gap(imgs).cpu())  # features reais do teacher
        feats_s.append(student(imgs).cpu())                # predicoes do student
        lbls.append(labels)
        total += len(imgs)

    ft = torch.cat(feats_t)[:n_amostras].numpy()
    fs = torch.cat(feats_s)[:n_amostras].numpy()
    lb = torch.cat(lbls)[:n_amostras].numpy()

    # PCA para 50 dims antes do t-SNE (acelera muito sem perder estrutura)
    n_comp = min(50, ft.shape[1], fs.shape[1], n_amostras - 1)
    pca    = PCA(n_components=n_comp, random_state=42)
    ft_r   = pca.fit_transform(ft)
    fs_r   = pca.fit_transform(fs)

    print(f'Executando t-SNE ({n_amostras} amostras)...')
    tsne   = TSNE(n_components=2, perplexity=30, random_state=42,
                  n_iter=1000, init='pca')
    ft_2d  = tsne.fit_transform(ft_r)
    fs_2d  = tsne.fit_transform(fs_r)

    # usa no maximo 20 cores distintas (tab20) para classes
    cores_cls = lb % 20

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    kw = dict(s=10, alpha=0.65, cmap='tab20', vmin=0, vmax=19)

    axes[0].scatter(ft_2d[:, 0], ft_2d[:, 1], c=cores_cls, **kw)
    axes[0].set_title(f't-SNE — Features TEACHER\n(resnet50, {dataset_nome})',
                      fontsize=11)
    axes[0].axis('off')

    axes[1].scatter(fs_2d[:, 0], fs_2d[:, 1], c=cores_cls, **kw)
    axes[1].set_title(f't-SNE — Predicoes STUDENT\n({best_encoder_por_teacher["resnet50"]}'
                      f', {dataset_nome})', fontsize=11)
    axes[1].axis('off')

    plt.suptitle(
        'Espaco de features: Teacher vs Student\n'
        'Clusters similares indicam boa transferencia de conhecimento (KD)',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ── executa t-SNE para resnet50 nos dois datasets ─────────────────────────────
for ds in DATASETS_NOMES:
    t_tsne = resultados_fase1['resnet50'][ds]['teacher_obj'].to(device)
    s_tsne = resultados_final['resnet50'][ds]['student_obj'].to(device)
    plotar_tsne(s_tsne, t_tsne, DATASETS[ds]['test'],
                n_amostras=500, dataset_nome=ds)
    t_tsne.cpu(); s_tsne.cpu()
